# Atividade Somativa 2
## Pipeline de Machine Learning para Previsão de Ruído Aerodinâmico
**Autor:** Lucas Schneider Cordeiro  
**Dataset Selecionado:** NASA Airfoil Self-Noise (`nasa.csv`)  

---

## 1. Introdução e Contextualização

Este trabalho representa a continuidade da Parte I. O objetivo central permanece sendo a previsão do nível de pressão sonora (`scaled-sound-pressure`) medido em decibéis, o que caracteriza este projeto como um problema de **Regressão**.

Nesta segunda fase, conforme as instruçõeso, o fluxo de desenvolvimento foi reestruturado. A ordem de execução dos passos foi ajustada para:

1. **Carregar o dataset** (o mesmo utilizado na Parte I).
2. **Realizar a divisão do dataset** entre uma base de treinamento e de testes (isolando os dados de teste logo no início).
3. **Realizar o passo de preparação dos dados** (aplicado dentro de um fluxo seguro).
4. **Realizar o treinamento do algoritmo** com um modelo de aprendizagem supervisionada (Random Forest).
5. **Mostrar as predições** para a base de testes.

A inversão da divisão dos dados para acontecer antes da preparação épara evitar o **vazamento de dados** (*data leakage*), garantindo que nenhuma informação estatística da base de teste contamine a base de treinamento. Para isso, utilizaremos a estrutura de `Pipeline`.

In [1]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn import set_config

# Carregamento do dataset utilizando o separador correto
df = pd.read_csv('nasa.csv', sep=';')

# Seleção prévia de atributos: Remoção da coluna redundante 'attack-angle'
# Conforme identificado na Parte I, esta variável apresenta alta correlação com as demais características físicas.
X = df.drop(columns=['scaled-sound-pressure', 'attack-angle'])
y = df['scaled-sound-pressure']

print(f"Dataset carregado com sucesso. Dimensões dos atributos de entrada (X): {X.shape}")

Dataset carregado com sucesso. Dimensões dos atributos de entrada (X): (1503, 4)


## 2. Divisão do Dataset (Treino e Teste)

A divisão é realizada nesta etapa utilizando a proporção metodológica exigida de 75% dos dados para treinamento e 25% para teste. 

Ao isolar a base de teste antes de aplicar a normalização matemática, garantimos a integridade do processo de avaliação, impedindo que informações da base de teste influenciem a escala dos dados de treino.

In [2]:
# Divisão dos dados em treino (75%) e teste (25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Quantidade de registros para Treinamento: {X_train.shape[0]}")
print(f"Quantidade de registros para Teste: {X_test.shape[0]}")

Quantidade de registros para Treinamento: 1127
Quantidade de registros para Teste: 376


## 3. Construção e Visualização do Pipeline

Para garantir que a preparação dos dados e o treinamento do algoritmo ocorram de forma sequencial e reproduzível, criamos um **Pipeline** que inclui os seguintes passos:
1. **Preparação dos Dados (`scaler`):** Aplicação do `StandardScaler` para normalizar as variáveis, garantindo que grandezas maiores não dominem o algoritmo.
2. **Treinamento do Algoritmo (`regressor`):** Instanciação do algoritmo supervisionado *Random Forest Regressor* (o mesmo utilizado na Parte I).

Abaixo, além de treinar o modelo com os dados de treino, renderizamos a arquitetura visual do pipeline.

In [3]:
# Definição dos passos do Pipeline
passos_pipeline = [
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
]

# Construção do Pipeline
pipeline_ml = Pipeline(steps=passos_pipeline)

# Treinamento do pipeline completo utilizando a base de treino
pipeline_ml.fit(X_train, y_train)

# Ativando a visualização gráfica (diagrama) nativa do Jupyter
set_config(display='diagram')

# Chamando a variável pipeline_ml isolada para gerar o gráfico interativo
pipeline_ml

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split 

## 4. Predições e Avaliação do Modelo

Com o pipeline devidamente consolidado, aplicamos os dados de teste desconhecidos para gerar as predições. Para mensurar o desempenho de forma analítica, selecionamos duas métricas de avaliação voltadas para regressão:

1. **MAE (Mean Absolute Error - Erro Absoluto Médio):** Mede a magnitude média dos erros nas previsões. É uma métrica linear e de fácil interpretação física (expressa diretamente em decibéis).
2. **RMSE (Root Mean Squared Error - Raiz do Erro Quadrático Médio):** Penaliza de forma mais severa os erros de maior magnitude devido à elevação ao quadrado, sendo ideal para identificar se o modelo está cometendo desvios críticos.

In [4]:
# Geração das predições utilizando a base de teste através do pipeline
y_pred = pipeline_ml.predict(X_test)

# Cálculo das métricas de desempenho solicitadas
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"Métricas de Avaliação Resultantes:")
print(f"1. Erro Absoluto Médio (MAE): {mae:.2f} dB")
print(f"2. Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f} dB\n")

# Visualização das 10 primeiras previsões lado a lado com os valores reais
tabela_comparativa = pd.DataFrame({
    'Valor Real (dB)': y_test.values,
    'Predição do Pipeline (dB)': np.round(y_pred, 2)
})
display(tabela_comparativa.head(10))

Métricas de Avaliação Resultantes:
1. Erro Absoluto Médio (MAE): 1.33 dB
2. Raiz do Erro Quadrático Médio (RMSE): 1.84 dB



,Valor Real (dB),Predição do Pipeline (dB)
0,125.045,123.44
1,118.767,119.27
2,120.233,119.15
3,137.047,136.55
4,134.556,134.26
5,120.766,123.55
6,124.525,123.82
7,136.798,134.55
8,130.123,133.44
9,123.991,128.19


## 5. Conclusões Finais e Análise das Métricas

### Interpretação dos Resultados
Os resultados confirmam que a mudança para o Pipeline funcionou muito bem. Conseguimos organizar melhor as etapas do projeto e evitar o vazamento de dados, mantendo a alta precisão do modelo. Analisando as métricas:
* O **MAE** indica que, em média, as estimativas de pressão sonora do algoritmo se afastam apenas cerca de **1,3 dB** em relação aos valores medidos fisicamente no túnel de vento. Em uma escala de ruído aeronáutico, essa variação é mínima e indica excelente capacidade preditiva.
* O **RMSE** manteve-se próximo ao valor do MAE, comprovando que o erro do modelo é estável e que não ocorreram predições com desvios extremos na base de teste.

---

### Proposta de Melhoria Futura: Otimização com GridSearchCV

Como passo futuro para escalar este projeto em um ambiente de produção, a principal melhoria seria realizar a otimização dos hiperparâmetros da Floresta Aleatória. 

Como agora nossa estrutura está encapsulada em um Pipeline, podemos realizar uma busca em grade (Grid Search) com Validação Cruzada (Cross-Validation) de forma robusta e limpa. Abaixo, apresento a implementação prática de como essa otimização pode ser executada para buscar o melhor número de árvores e a melhor profundidade para o modelo:

In [5]:
# 1. Definimos o dicionário de parâmetros a serem testados.
# Nota: O prefixo 'regressor__' é necessário para indicar que o parâmetro pertence ao Random Forest dentro do Pipeline
param_grid = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [None, 10, 20]
}

# 2. Instanciamos o GridSearchCV passando o Pipeline inteiro
# cv=5 significa que usaremos 5-fold cross-validation
otimizador = GridSearchCV(pipeline_ml, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

# 3. Executamos a busca com os dados de treino
print("Iniciando otimização de hiperparâmetros...")
otimizador.fit(X_train, y_train)

print("\nOtimização Concluída!")
print(f"Melhores parâmetros encontrados: {otimizador.best_params_}")

Iniciando otimização de hiperparâmetros...

Otimização Concluída!
Melhores parâmetros encontrados: {'regressor__max_depth': None, 'regressor__n_estimators': 200}


### Conclusão da Otimização
A busca em grade com validação cruzada identificou com sucesso a melhor configuração para o nosso modelo preditivo:
* **`n_estimators`: 200** – O modelo obteve sua melhor performance utilizando 200 árvores de decisão simultâneas. Um número maior de árvores aumenta a robustez e a estabilidade da "Floresta", diluindo o viés de árvores individuais.
* **`max_depth`: None** – O algoritmo performou melhor sem um limite artificial de profundidade para as árvores. Isso indica que as relações físicas entre o vento e o ruído são complexas o suficiente para exigir expansão total dos nós de decisão. Graças à natureza do *Random Forest* e ao controle da validação cruzada, essa profundidade total foi alcançada sem gerar *overfitting* (superajuste).

Com essa otimização, o Pipeline está perfeitamente ajustado e pronto para ser implementado em um ambiente de produção para simulações aerodinâmicas reais.